<a href="https://colab.research.google.com/github/bintangbig-tech/chatbot-kehadiran-pejabat-samarinda-ilir/blob/main/Chatbox_SMR_ILIR.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -qU langchain langchain-google-genai langchain-core

In [ ]:
import os
from google.colab import userdata
from langchain_core.tools import tool
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage

# Membaca API Key Gemini dari Colab Secrets
try:
    os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
except Exception:
    print("❌ GOOGLE_API_KEY belum diset di menu Secrets.")

# 2. Integrasi API Eksternal (Simulasi Tool)
@tool
def cek_kehadiran(jabatan: str) -> str:
    """Mengecek status kehadiran pejabat di Kecamatan Samarinda Ilir berdasarkan jabatan."""
    data_absensi = {
        "camat": {"status": "Hadir", "waktu_masuk": "07:45"},
        "sekcam": {"status": "Dinas Luar", "waktu_masuk": "-"},
        "kasi trantib": {"status": "Hadir", "waktu_masuk": "08:10"},
        "kasi pemerintahan": {"status": "Hadir", "waktu_masuk": "08:15"}
    }

    jabatan_key = jabatan.lower().strip()
    if jabatan_key in data_absensi:
        info = data_absensi[jabatan_key]
        return f"Status {jabatan}: {info['status']}, Jam: {info['waktu_masuk']}"
    return f"Data presensi untuk {jabatan} tidak ditemukan di sistem."

# 3. Konfigurasi LLM (Menggunakan Gemini 1.5 Flash)
llm = ChatGoogleGenerativeAI(model="gemini-1.5-flash", temperature=0.7)
tools = [cek_kehadiran]
llm_with_tools = llm.bind_tools(tools)

# 4. Konfigurasi Karakter
prompt_template = ChatPromptTemplate.from_messages([
    ("system", "Kamu adalah asisten virtual Kecamatan Samarinda Ilir. Gunakan bahasa yang santai, luwes, namun tetap sopan. "
               "Selalu sapa pejabat dengan sebutan Bapak/Ibu. Jika ditanya soal kehadiran, langsung gunakan tool cek_kehadiran."),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}"),
])

# 5. Custom Execution Loop (Logika yang Anda buat)
def run_agent(user_input, history):
    formatted_prompt = prompt_template.format_messages(input=user_input, chat_history=history)
    response = llm_with_tools.invoke(formatted_prompt)

    if response.tool_calls:
        # Jika butuh panggil tool, siapkan pesan untuk ronde kedua
        messages_for_second_round = formatted_prompt + [response]

        for tool_call in response.tool_calls:
            if tool_call["name"] == "cek_kehadiran":
                tool_output = cek_kehadiran.invoke(tool_call["args"])
                messages_for_second_round.append(ToolMessage(content=tool_output, tool_call_id=tool_call["id"]))

        # Panggil kembali model setelah tool memberikan hasil
        second_response = llm_with_tools.invoke(messages_for_second_round)

        # Simpan ke history utama
        history.append(HumanMessage(content=user_input))
        history.append(second_response)
        return second_response.content
    else:
        history.append(HumanMessage(content=user_input))
        history.append(response)
        return response.content

# --- UJI COBA CHATBOT INTERAKTIF ---
chat_history = []

print("✅ Chatbot Kecamatan Samarinda Ilir Siap!")
print("Ketik pertanyaan Anda di bawah ini (ketik 'keluar' untuk berhenti).\n")
print("=" * 60)

while True:
    pesan_user = input("Anda: ")
    if pesan_user.lower() == 'keluar':
        print("Chatbot: Terima kasih, selamat bertugas kembali!")
        break

    balasan = run_agent(pesan_user, chat_history)
    print(f"Chatbot: {balasan}\n")

✅ Chatbot Kecamatan Samarinda Ilir Siap!
Ketik pertanyaan Anda di bawah ini (ketik 'keluar' untuk berhenti).

